In [ ]:
pip install Korpora

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
df = pd.DataFrame(corpus.test).sample(20000, random_state=42)
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(train.head(5).to_markdown())
print(f"Training Data Size : {len(train)}")
print(f"Validation Data Size : {len(valid)}")
print(f"Testing Data Size : {len(test)}")



    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
import torch
from transformers import BertTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(
        text=data.text.tolist(),
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )
    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)
    labels = torch.tensor(data.label.values, dtype=torch.long).to(device)
    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

# 설정
epochs = 5
batch_size = 32
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    do_lower_case=False
)

# 데이터셋 및 데이터로더 생성
train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

(tensor([   101,  58466,   9812, 118956, 119122,  59095,  10892,   9434, 118888,
           117,   9992,  40032,  30005,    117,   9612,  37824,   9410,  12030,
         42337,  10739,  83491,  12508,    106,    106,    102,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,

In [ ]:
from torch import optim
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import numpy as np
import torch
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss = 0.0
        val_accuracy = 0.0

        for input_ids, attention_mask, labels in dataloader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            logits = outputs.logits
            loss = criterion(logits, labels)

            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss.item()
            val_accuracy += accuracy

        val_loss = val_loss / len(dataloader)
        val_accuracy = val_accuracy / len(dataloader)

        return val_loss, val_accuracy

# 학습 루프
best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)

    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy: {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "BRET.pt")
        print("Saved the model weights")


Epoch 1: Train Loss: 0.4117 Val Loss: 0.4175 Val Accuracy: 0.8093
Saved the model weights
Epoch 2: Train Loss: 0.3277 Val Loss: 0.4329 Val Accuracy: 0.8067
Epoch 3: Train Loss: 0.2555 Val Loss: 0.4795 Val Accuracy: 0.8143
Epoch 4: Train Loss: 0.1922 Val Loss: 0.5138 Val Accuracy: 0.8103
Epoch 5: Train Loss: 0.1488 Val Loss: 0.5318 Val Accuracy: 0.8097


In [ ]:
from transformers import BertForSequenceClassification

# 학습한 BERT 모델 로드
model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)

# 학습된 가중치 불러오기
model.load_state_dict(torch.load("BRET.pt"))

# 테스트 데이터 평가
test_loss, test_accuracy = evaluation(model, test_dataloader)
print(f"Test Loss : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Loss : 0.4151
Test Accuracy : 0.8127


# BART

In [ ]:
pip install datasets

In [ ]:
!rm -rf ~/.cache/huggingface/datasets/argilla__news-summary

In [ ]:
import numpy as np
import pandas as pd

# Hugging Face Parquet 경로
splits = {
    'train': 'data/train-00000-of-00001-ebc48879f34571f6.parquet',
    'test': 'data/test-00000-of-00001-6227bd8eb10a9b50.parquet'
}

# test split에서 읽기
df = pd.read_parquet("hf://datasets/argilla/news-summary/" + splits["test"])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import numpy as np

# 1. 요약이 문자열로 들어 있는 경우 eval 처리 (nested 구조 대응)
df["prediction"] = df["prediction"].apply(
    lambda x: eval(x)[0]["text"] if isinstance(x, str) and x.startswith("[{") else x
)

# 2. 5000개 샘플링
df = df.sample(5000, random_state=42)

# 3. 다시 리스트 형태에서 텍스트만 꺼내기
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])

# 4. Train / Validation / Test 분할
train, valid, test = np.split(
    df.sample(frac=1, random_state=42),
    [int(0.6 * len(df)), int(0.8 * len(df))]
)

# 5. 출력
print(f"Source News : {train.text.iloc[0][:200]}")
print(f"Summarization : {train.prediction.iloc[0][:50]}")
print(f"Training Data Size : {len(train)}")
print(f"Validation Data Size : {len(valid)}")
print(f"Testing Data Size : {len(test)}")


In [ ]:
import torch
from transformers import BartTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(
        text=data.text.tolist(),
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)

    labels = []
    for target in data.prediction:
        labels.append(tokenizer.encode(target, return_tensors="pt").squeeze())

    labels = pad_sequence(labels, batch_first=True, padding_value=-100).to(device)

    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

# 하이퍼파라미터 설정
epochs = 3
batch_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BartTokenizer.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
)

# 데이터셋 및 데이터로더 생성
train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])


(tensor([    0,   495,  1889,  9298,     6,  5490,    36,  1251,    43,   111,
         1083,   270,  6546,  3176,    26,    15,   378,    37,    56,    10,
         2340,  6054,    19,   121,     4,   104,     4,   884,   807,   140,
           23,    10,  3564,    11,  5490,     6,     8,  1602,   140,    25,
         2366,     6,   157,    12, 26414,     6,     8,  3473,     7,   432,
           19,     4,  3176,    26,    14,    10, 30036,   196,  9526,  2662,
           12,  3955,   529,    19,   140,   222,    45,  1369,    23,     5,
         1817,    12,  8145,  4713, 18204,  3564,     6,  4319, 19114,   743,
           15,   258,  2380,     8, 20022, 11883,   743,     4,  3176,     6,
           23,    10,  7515,    13,  1865,    23,     5,   253,     9,     5,
         3564,     6,    26,    89,    21,   202,    10,   240,    13,   617,
          121,     4,   104,  3358, 15685,  9872,     6,   258,    23,     5,
          672,     9,  3885,     9,   194,     8,    49,   503,

In [ ]:
##허깅 페이스 모델 평가 라이브러리
!pip install evaluate rouge_score absl-py

In [ ]:
import numpy as np
import torch
from torch import nn
import evaluate

# ROUGE 스코어 계산 함수
def calc_rouge(preds, labels):
    preds = preds.argmax(axis=-1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    rouge2 = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    return rouge2["rouge2"]

# 학습 함수
def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

# 평가 함수
def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0
        val_rouge = 0.0

        for input_ids, attention_mask, labels in dataloader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            logits = outputs.logits
            loss = outputs.loss

            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            rouge = calc_rouge(logits, label_ids)

            val_loss += loss.item()
            val_rouge += rouge

        val_loss = val_loss / len(dataloader)
        val_rouge = val_rouge / len(dataloader)
        return val_loss, val_rouge

# ROUGE 평가기 초기화
rouge_score = evaluate.load("rouge", tokenizer=tokenizer)


In [ ]:
from transformers import BartForConditionalGeneration
from torch import optim

# 디바이스 설정 (GPU 사용 가능하면 사용)
device = "cuda" if torch.cuda.is_available() else "cpu"

# BART 모델 불러오기
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)

# 옵티마이저 설정
optimizer = optim.AdamW(model.parameters(), lr=5e-5, eps=1e-8)


In [ ]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_rouge = evaluation(model, valid_dataloader)

    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} ROUGE-2: {val_rouge:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "BART.pt")
        print("Saved the model weights")


Epoch 1: Train Loss: 2.1503 Val Loss: 1.8627 ROUGE-2: 0.2579
Saved the model weights
Epoch 2: Train Loss: 1.6108 Val Loss: 1.8878 ROUGE-2: 0.2598
Epoch 3: Train Loss: 1.2267 Val Loss: 1.9929 ROUGE-2: 0.2495


In [ ]:
from transformers import BartForConditionalGeneration

# 모델 로드
model = BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)

# 저장된 가중치 불러오기 (경로 수정)
model.load_state_dict(torch.load("BART.pt"))

# 테스트 평가
test_loss, test_rouge_score = evaluation(model, test_dataloader)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test ROUGE-2 Score : {test_rouge_score:.4f}")


Test Loss : 1.8220
Test ROUGE-2 Score : 0.2613


#ELECTRA

In [ ]:
import torch
from transformers import ElectraTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(
        text=data.text.tolist(),
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)
    labels = torch.tensor(data.label.values, dtype=torch.long).to(device)

    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader


In [ ]:
# 설정
epochs = 5
batch_size = 32
device = "cuda" if torch.cuda.is_available() else "cpu"

# ELECTRA 토크나이저
tokenizer = ElectraTokenizer.from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    do_lower_case=False
)

# 데이터셋 및 데이터로더
train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])


(tensor([    2,  6511, 14347,  4087,  4665,  4112,  2924,  4806,    16,  3809,
         4309,  4275,    16,  3201,  4376,  2891,  4139,  4212,  4007,  6557,
         4200,     5,     5,     3,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0], device='cuda:0'), tensor([1, 1, 1, 1, 1, 1, 1, 

In [ ]:
from torch import optim
from transformers import ElectraForSequenceClassification

model = ElectraForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)


pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import ElectraForSequenceClassification

model = ElectraForSequenceClassification.from_pretrained(
    "monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)

# 전체 모델 구조 출력
print(model)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(35000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [ ]:
import numpy as np
import torch
from torch import nn

# 정확도 계산 함수
def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

# 학습 함수
def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return train_loss / len(dataloader)

# 평가 함수
def evaluate(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0
        val_accuracy = 0.0
        criterion = nn.CrossEntropyLoss()

        for input_ids, attention_mask, labels in dataloader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            logits = outputs.logits
            loss = criterion(logits, labels)

            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            acc = calc_accuracy(logits, label_ids)

            val_loss += loss.item()
            val_accuracy += acc

        return val_loss / len(dataloader), val_accuracy / len(dataloader)


In [ ]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluate(model, valid_dataloader)

    print(f"Epoch {epoch + 1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, Val Accuracy = {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "ELECTRA.pt")
        print("Saved the model weights")


Epoch 1: Train Loss = 0.6955, Val Loss = 0.6953, Val Accuracy = 0.4698
Saved the model weights
Epoch 2: Train Loss = 0.6955, Val Loss = 0.6953, Val Accuracy = 0.4698
Epoch 3: Train Loss = 0.6955, Val Loss = 0.6953, Val Accuracy = 0.4698
Epoch 4: Train Loss = 0.6954, Val Loss = 0.6953, Val Accuracy = 0.4698
Epoch 5: Train Loss = 0.6954, Val Loss = 0.6953, Val Accuracy = 0.4698


In [ ]:
# 저장된 KoELECTRA 모델 로드
from transformers import ElectraForSequenceClassification

model = ElectraForSequenceClassification.from_pretrained(
    "monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)

model.load_state_dict(torch.load("ELECTRA.pt"))

# 테스트 데이터 평가
test_loss, test_accuracy = evaluate(model, test_dataloader)
print(f"Test Loss = {test_loss:.4f}")
print(f"Test Accuracy = {test_accuracy:.4f}")


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Loss = 0.6953
Test Accuracy = 0.4627


#T5

In [ ]:
import numpy as np

# 이미 pandas DataFrame 이므로 .to_pandas() 제거
df = df[["text", "prediction"]].sample(5000, random_state=42)

# prediction 열이 ndarray이면 첫 번째 요소로 변환
df["prediction"] = df["prediction"].apply(lambda x: str(x[0]) if isinstance(x, (list, np.ndarray)) else str(x))

# T5 입력 prefix 붙이기
df["text"] = df["text"].apply(lambda x: x if x.startswith("summarize: ") else "summarize: " + x)

# Train / Valid / Test 분할
train, valid, test = np.split(
    df.sample(frac=1, random_state=42),
    [int(0.6 * len(df)), int(0.8 * len(df))]
)

# 확인용 출력
print(f"Source News : {train.text.iloc[0][:200]}")
print(f"Summarization : {train.prediction.iloc[0][:50]}")
print(f"Training Data Size : {len(train)}")
print(f"Validation Data Size : {len(valid)}")
print(f"Testing Data Size : {len(test)}")


Source News : summarize: summarize: (Reuters) - Hillary Clinton agreed on Thursday to some of the terms laid down by an opponent, Senator Bernie Sanders, in his call to increase the number of public debates as they
Summarization : {'score': 1.0, 'text': 'Hillary Clinton willing to
Training Data Size : 3000
Validation Data Size : 1000
Testing Data Size : 1000


/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
    # 문자열 리스트로 변환
    text_list = data["text"].tolist()
    label_list = data["prediction"].tolist()

    # 문자열이 아니라 리스트 등으로 잘못 들어간 경우 방어 처리
    text_list = [t if isinstance(t, str) else str(t) for t in text_list]
    label_list = [l[0]["text"] if isinstance(l, list) and isinstance(l[0], dict) else l for l in label_list]

    # prefix 붙이기
    text_list = ["summarize: " + t for t in text_list]

    source = tokenizer(
        text_list,
        padding="max_length",
        max_length=128,
        truncation=True,
        return_tensors="pt"
    )

    target = tokenizer(
        label_list,
        padding="max_length",
        max_length=128,
        truncation=True,
        return_tensors="pt"
    )

    source_ids = source["input_ids"].to(device)
    source_mask = source["attention_mask"].to(device)
    target_ids = target["input_ids"].to(device)
    target_mask = target["attention_mask"].to(device)

    return TensorDataset(source_ids, source_mask, target_ids, target_mask)



In [ ]:
epochs = 3
batch_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = T5Tokenizer.from_pretrained("t5-small")


In [ ]:
print(train.text.tolist()[:3])

['summarize: summarize: (Reuters) - Hillary Clinton agreed on Thursday to some of the terms laid down by an opponent, Senator Bernie Sanders, in his call to increase the number of public debates as they vie to become the Democratic candidate in November’s U.S. presidential election. Clinton’s two main Democratic challengers, Sanders and former Maryland Governor Martin O’Malley, have long called for more debates. They have complained that the relatively skimpy schedule of only six encounters was designed by the party to protect Clinton’s position at the top of opinion polls. But the call for more debates intensified this week after a hastily arranged debate next Thursday in New Hampshire was announced, organized by a news channel and a state newspaper. Both Clinton and O’Malley said they would attend if all the candidates agreed, despite risking the ire of the Democratic National Committee, which has forbidden candidates from taking part in unsanctioned encounters.  On Wednesday evening

In [ ]:
print(type(df.text.iloc[0]))  # 반드시 str 이어야 함
print(type(df.prediction.iloc[0]))  # 이게 list 이면 처리 필요

<class 'str'>
<class 'str'>


In [ ]:
train_dataset = make_dataset(train, tokenizer, device)
valid_dataset = make_dataset(valid, tokenizer, device)
test_dataset  = make_dataset(test, tokenizer, device)

train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)
test_dataloader  = get_dataloader(test_dataset, SequentialSampler, batch_size)


In [ ]:
print(next(iter(train_dataloader)))  # 배치 하나 확인
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))

[tensor([[21603,    10, 21603,  ...,     3, 29155,     1],
        [21603,    10, 21603,  ...,   809,    11,     1],
        [21603,    10, 21603,  ...,   398,   888,     1],
        ...,
        [21603,    10, 21603,  ...,   385,   315,     1],
        [21603,    10, 21603,  ...,    30,  2875,     1],
        [21603,    10, 21603,  ...,     0,     0,     0]], device='cuda:0'), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'), tensor([[ 3,  2, 31,  ...,  0,  0,  0],
        [ 3,  2, 31,  ...,  0,  0,  0],
        [ 3,  2, 31,  ...,  0,  0,  0],
        ...,
        [ 3,  2, 31,  ...,  0,  0,  0],
        [ 3,  2, 31,  ...,  0,  0,  0],
        [ 3,  2, 31,  ...,  0,  0,  0]], device='cuda:0'), tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        .

In [ ]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(
    "t5-small"
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

# 전체 모델 구조 출력
print(model)


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
import numpy as np
from torch import nn

# 학습 함수
def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(
            input_ids=source_ids,
            attention_mask=source_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return train_loss / len(dataloader)

# 평가 함수
def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0

        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()
            labels = target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

            outputs = model(
                input_ids=source_ids,
                attention_mask=source_mask,
                decoder_input_ids=decoder_input_ids,
                labels=labels,
            )

            loss = outputs.loss
            val_loss += loss.item()

    return val_loss / len(dataloader)


In [ ]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)

    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "T5.pt")
        print("Saved the model weights")


Epoch 1: Train Loss: 5.3806 Val Loss: 4.8962
Saved the model weights
Epoch 2: Train Loss: 5.3781 Val Loss: 4.8962
Epoch 3: Train Loss: 5.3866 Val Loss: 4.8962


In [ ]:
model.eval()
with torch.no_grad():
    for source_ids, source_mask, target_ids, target_mask in test_dataloader:
        # 문장 생성
        generated_ids = model.generate(
            input_ids=source_ids,
            attention_mask=source_mask,
            max_length=128,
            num_beams=3,
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True,
        )

        # 생성 결과와 실제 타깃 비교
        for generated, target in zip(generated_ids, target_ids):
            pred = tokenizer.decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )
            actual = tokenizer.decode(
                target,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )
            print("Generated Headline Text:", pred)
            print("Actual Headline Text   :", actual)
        break  # 배치 하나만 실행


Generated Headline Text: a fractured vote brings the far-right into parliament means she must try to work out a three-way coalition untested at federal level. the alliance would comprise Merkel s conservative bloc and the Bavarian Christian Social Union.
Actual Headline Text   : 'score': 1.0, 'text': "Factbox: German coalition watch - Let's not be perfectionists in coalition talks, says Merkel ally"
Generated Headline Text: u.s. defense secretary says there is no indication that Russia wants a positive relationship with the united states. "that is not to say we can’t get there as we look for common ground," he said.
Actual Headline Text   : 'score': 1.0, 'text': 'No indication Russia wants positive relationship with U.S.: Mattis'
Generated Headline Text: parties have been unable to focus on the elections because of turmoil that followed a referendum on sept. 25. in baghdad as well as neighbors Iran and Turkey opposed the referendum that saw a clear independence.
Actual Headline Text   